# 🟢 Activity 07: Linear Regression with TensorFlow / Keras

**Track:** Linear Regression  
**Level:** Intermediate → Advanced

---

## 📖 What You Will Learn
- Why use TensorFlow for linear regression?
- Build a linear model with `tf.keras`
- Understand Dense layers as linear transformations
- Track and visualise training/validation loss
- Use callbacks: EarlyStopping, ReduceLROnPlateau
- Compare with sklearn results

---

## 🧠 Concept: Linear Regression as a Neural Network

A single Dense layer with no activation function is exactly linear regression:

```
Input (p features) → Dense(1, activation=None) → Output (prediction)
```

TF minimises MSE using gradient descent (same math as notebook 05),
but with automatic differentiation and GPU acceleration.

> **Why TF for simple linear regression?**  
> You're practising the deep learning workflow that scales to complex architectures.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression as SklearnLR
from sklearn.metrics import r2_score, mean_squared_error

import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 📊 Data Preparation

In [ ]:
housing = fetch_california_housing(as_frame=True)
X = housing.frame.drop(columns=['MedHouseVal'])
y = housing.frame['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train).astype(np.float32)
X_te_sc = scaler.transform(X_test).astype(np.float32)
y_tr    = y_train.values.astype(np.float32)
y_te    = y_test.values.astype(np.float32)

print(f'Train: {X_tr_sc.shape}  Test: {X_te_sc.shape}')

## 🏗️ Model 1 — Simple Linear Model (Single Dense Layer)

In [ ]:
def build_linear_model(input_dim, lr=0.001):
    """
    A single Dense layer with no activation = Linear Regression.
    WHY: This is the foundation. Every complex network is built from the same principles.
    """
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(1)          # No activation → y = Wx + b
    ], name='LinearRegression')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='mse',
        metrics=['mae']
    )
    return model


model = build_linear_model(input_dim=X_tr_sc.shape[1])
model.summary()

In [ ]:
# ── Callbacks ──────────────────────────────────────────────────────────────────
# WHY EarlyStopping?
# Training too long → overfitting. EarlyStopping monitors val_loss and stops when it stops improving.
# restore_best_weights=True ensures we keep the best checkpoint, not the final (overfit) one.

early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_tr_sc, y_tr,
    validation_split=0.2,
    epochs=300,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=0             # suppress per-epoch output; use history instead
)

print(f'\nTraining stopped at epoch {len(history.history["loss"])}')

## 📈 Loss Curve Visualisation

In [ ]:
def plot_loss_curves(history, title='Training History'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history.history['loss'],     label='Train MSE')
    axes[0].plot(history.history['val_loss'], label='Val MSE')
    axes[0].set_title(f'{title} — MSE Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MSE')
    axes[0].legend()
    axes[0].set_yscale('log')

    axes[1].plot(history.history['mae'],     label='Train MAE')
    axes[1].plot(history.history['val_mae'], label='Val MAE')
    axes[1].set_title(f'{title} — MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_loss_curves(history, 'Linear Model')

# Evaluate
y_pred_tf = model.predict(X_te_sc, verbose=0).flatten()
r2_tf   = r2_score(y_te, y_pred_tf)
rmse_tf = np.sqrt(mean_squared_error(y_te, y_pred_tf))
print(f'TF Linear Model — R²: {r2_tf:.4f}  RMSE: {rmse_tf:.4f}')

## 🏗️ Model 2 — Deep Neural Network (Non-linear)

### 🧠 Concept: Going Beyond Linear

Adding hidden layers + non-linear activations makes the model a **Universal Function Approximator** —
it can learn any continuous function, including non-linear relationships in housing prices.

In [ ]:
def build_deep_model(input_dim, lr=0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.1),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ], name='DeepRegressor')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='huber',   # Huber loss: less sensitive to outliers than MSE
        metrics=['mae']
    )
    return model


deep_model = build_deep_model(input_dim=X_tr_sc.shape[1])
deep_model.summary()

In [ ]:
deep_history = deep_model.fit(
    X_tr_sc, y_tr,
    validation_split=0.2,
    epochs=200,
    batch_size=64,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6)
    ],
    verbose=0
)

plot_loss_curves(deep_history, 'Deep Model')

y_pred_deep = deep_model.predict(X_te_sc, verbose=0).flatten()
r2_deep   = r2_score(y_te, y_pred_deep)
rmse_deep = np.sqrt(mean_squared_error(y_te, y_pred_deep))
print(f'Deep Model — R²: {r2_deep:.4f}  RMSE: {rmse_deep:.4f}')

## 📊 Comparison: sklearn vs TF Linear vs TF Deep

In [ ]:
# sklearn benchmark
sklearn_lr = SklearnLR()
sklearn_lr.fit(X_tr_sc, y_tr)
r2_sk = r2_score(y_te, sklearn_lr.predict(X_te_sc))

models = ['sklearn (OLS)', 'TF Linear', 'TF Deep (MLP)']
scores = [r2_sk, r2_tf, r2_deep]

plt.figure(figsize=(8, 4))
colors = ['steelblue', 'tomato', 'limegreen']
bars = plt.bar(models, scores, color=colors, edgecolor='white')
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{score:.4f}', ha='center', fontsize=10)
plt.ylabel('Test R²')
plt.title('R² Comparison: sklearn vs TensorFlow Models')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

print('\n💡 The deep model captures non-linearities that pure linear regression misses.')

## 💾 Save & Load Model

In [ ]:
import os
os.makedirs('models', exist_ok=True)

# Save in native Keras format
deep_model.save('models/deep_regressor.keras')
print('✅ Model saved')

# Reload and verify
loaded_model = keras.models.load_model('models/deep_regressor.keras')
y_pred_loaded = loaded_model.predict(X_te_sc, verbose=0).flatten()
assert np.allclose(y_pred_deep, y_pred_loaded, atol=1e-5), 'Mismatch!'
print('✅ Model reloaded and predictions match')

## 🎯 Summary

| Model | Key TF Concepts | Test R² |
|---|---|---|
| Linear (Dense 1) | Same as OLS; trains with gradient descent | Similar to sklearn |
| Deep (MLP) | Hidden layers + ReLU + BatchNorm + Dropout | Higher (captures non-linearity) |

**Key Takeaways:**
- `EarlyStopping` + `restore_best_weights=True` is **mandatory** to avoid overfitting
- Always scale inputs before feeding to a neural network
- Huber loss is more robust than MSE when outliers are present
- BatchNorm + Dropout are standard regularisation in deep regression

---

**Next:** [08_linear_regression_pytorch.ipynb](08_linear_regression_pytorch.ipynb)